# Semantic Memory: Structured Personal Knowledge

In the previous notebook, we gave the agent **episodic memory** — the ability to remember
past events ("Sarah's flight to London was delayed three hours"). But events are raw
material, not actionable knowledge.

Ask the agent "What airline does Sarah prefer?" and it must scan dozens of trip records,
hoping to infer a pattern. That is expensive, unreliable, and produces a different answer
every time the model changes.

**Semantic memory** stores the conclusion directly: structured personal facts and preferences
with categories, confidence, and timestamps. It answers "what do we know?" rather than
"what happened?"

| Dimension | Episodic Memory | Semantic Memory |
|---|---|---|
| **Stores** | Events, experiences | Facts, preferences, beliefs |
| **Example** | "Flew United to London on 5 Jan" | "Prefers United Airlines" |
| **Query** | "What happened on the London trip?" | "What airline does Sarah prefer?" |
| **Schema** | Flat documents (event_type, description) | Structured documents (category, preference, confidence) |
| **Backend** | Cosmos DB (events container) | Cosmos DB (semantic-memory container) with vector search |
| **Reasoning** | Retrieve events, hope LLM connects the dots | Direct lookup with semantic similarity |

## The Problem: Episodic Memory Can't Answer "What Do We Know?"

Sarah Chen (employee E001) has accumulated trip records, feedback, and event logs in episodic memory. The agent can recall that she flew United twice and rated a Marriott 5 stars. But:

1. **Every recall requires inference** — the model must scan events and guess the preference
2. **Answers vary** — different model calls produce different conclusions from the same events
3. **No confidence signal** — is "prefers United" from one trip or twenty?
4. **No structure** — "vegetarian" is buried in a dinner feedback comment, not tagged as a dietary requirement
5. **No history** — if Sarah switches from Marriott to Hilton, both facts coexist with no priority

Semantic memory solves this by storing the **conclusion** as a structured document with metadata:

```json
{
  "category": "hotel_chain",
  "preference": "Prefers Marriott hotels",
  "confidence": 0.9,
  "state": "provisional",
  "valid_from": "2026-09-01",
  "valid_to": null
}
```

When the preference changes, the old document is retired (`valid_to` set) and the new one becomes current. Vector embeddings handle synonym resolution — "hotel preference", "where does Sarah stay", and "accommodation choice" all match the same document.

## Why Documents + Vector Search

Personal preferences are statements like "I prefer Marriott" or "Always book aisle seats". These are:

- **Self-contained** — each preference makes sense on its own; no multi-hop traversal needed
- **Synonymous** — "prefers Marriott", "always stays at Marriott", and "Marriott fan" mean the same thing
- **Evolving** — preferences change over time and need versioning

Vector embeddings solve the synonym problem at query time — no ontology, no relationship normalization, no NLP pipeline. And Cosmos DB gives us document storage, partial updates for lifecycle properties, native TTL, and DiskANN vector search in one service we already use for episodic memory.

In [ ]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Setup

We connect to Azure Cosmos DB (same service as episodic memory, separate container)
and Azure OpenAI for embeddings. The `SemanticMemoryStore` in `shared/semantic_store.py`
handles preference storage, vector search, and supersession.

In [ ]:
import json
import os
import sys

import certifi
import sniffio

sys.path.insert(0, "..")
sniffio.current_async_library_cvar.set("asyncio")
os.environ["SSL_CERT_FILE"] = certifi.where()

from dotenv import load_dotenv
from azure.identity import AzureCliCredential
from azure.identity.aio import (
    AzureCliCredential as AsyncCliCredential,
    get_bearer_token_provider as async_get_bearer_token_provider,
)
from azure.cosmos.aio import CosmosClient
from openai import AsyncAzureOpenAI
from agent_framework import Agent, AgentSession, tool
from shared.travel_agent import (
    SYSTEM_PROMPT,
    create_client,
    search_flights,
    search_hotels,
    get_travel_policy,
)
from shared.semantic_store import SemanticMemoryStore, create_container

load_dotenv("../.env", override=True)
client, credential = create_client("../.env")
print("Foundry client ready")

c:\Users\divyesheth\OneDrive - Microsoft\Documents\python-projects\agentic-memory\.venv\Lib\site-packages\agent_framework\_skills.py:122: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Users\divyesheth\OneDrive - Microsoft\Documents\python-projects\agentic-memory\.venv\Lib\site-packages\agent_framework\_harness\_file_access.py:602: ExperimentalWarning: [HARNESS] AgentFileStore is experimental and may change or be removed in future versions without notice.


Foundry client ready


In [ ]:
# Cosmos DB client
cosmos = CosmosClient(os.environ["COSMOS_ENDPOINT"], credential=AsyncCliCredential())
container = await create_container(cosmos)
print(f"Cosmos container ready: semantic-memory")

# Embedding function (uses whatever model .env specifies)
embed_deployment = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-ada-002")
embed_client = AsyncAzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_ad_token_provider=async_get_bearer_token_provider(
        AsyncCliCredential(), "https://cognitiveservices.azure.com/.default",
    ),
    api_version="2024-02-01",
)

async def embed(text: str) -> list[float]:
    r = await embed_client.embeddings.create(input=[text], model=embed_deployment)
    return r.data[0].embedding

store = SemanticMemoryStore(container, user_id="E001", embed_fn=embed)
await store.reset()
print(f"SemanticMemoryStore ready for E001 (embedding: {embed_deployment})")

LLM configured via Foundry: https://npmsfoundrysc.services.ai.azure.com/api/projects/proj-default
Embeddings configured via Azure OpenAI: https://npmsfoundrysc.openai.azure.com / text-embedding-ada-002


In [ ]:
## Storing Preferences

Each preference is a document with a category, text, confidence score, and vector embedding. The embedding is computed automatically from the preference text.

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nCALL db.index.vector.queryNodes('preference_embedding_idx', $limit, $embedding)\nYIELD node, score\nWHERE score >= $threshold\n  AND node.category = $category\nRETURN node AS p, score\nORDER BY score DESC\n"
Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replac

Stored 3 preferences (airline, hotel, seating) — linked to user E001


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nCALL db.index.vector.queryNodes('preference_embedding_idx', $limit, $embedding)\nYIELD node, score\nWHERE score >= $threshold\n  AND node.category = $category\nRETURN node AS p, score\nORDER BY score DESC\n"


Preference(id=UUID('930e4f52-821e-4116-911a-9e20df8fa4bf'), created_at=datetime.datetime(2026, 9, 7, 15, 56, 31, 957264), updated_at=None, embedding=[0.003027882194146514, -0.016573671251535416, -0.01165151409804821, -0.01716511882841587, -0.005316455382853746, 0.01769085042178631, -0.02593168430030346, -0.004951729439198971, -0.011257216334342957, 0.0024890080094337463, 0.019780630245804787, 0.012689832597970963, -0.004287994001060724, 0.0006008941563777626, -0.0012855767272412777, 0.002209713216871023, 0.022895587608218193, 0.011888093315064907, 0.02314530871808529, -0.006887076888233423, -0.021646976470947266, -0.004721722099930048, 0.010987779125571251, -0.026523131877183914, -0.007918823510408401, 0.02478821948170662, -0.00361440097913146, -0.021883554756641388, 0.003157672006636858, -0.0008304907823912799, 0.024617355316877365, 0.01426045410335064, -0.001833487069234252, -0.03703775256872177, -0.0038049784488976, -0.007287946529686451, -0.004419426433742046, 0.005237595643848181,

In [ ]:
await store.add_preference("airline", "I always fly United", confidence=0.8)
await store.add_preference("hotel_chain", "Prefers Marriott hotels", confidence=0.9)
await store.add_preference("seating", "Always requests aisle seats", confidence=0.85)
await store.add_preference("dietary", "Strict vegetarian — no meat in meals", confidence=0.95)
await store.add_preference("home_city", "Based in New York", confidence=0.8)
await store.add_preference("loyalty", "Star Alliance Gold member", confidence=0.85)

print(f"Stored {await store.count()} preferences for E001")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nCALL db.index.vector.queryNodes('preference_embedding_idx', $limit, $embedding)\nYIELD node, score\nWHERE score >= $threshold\n  AND node.category = $category\nRETURN node AS p, score\nORDER BY score DESC\n"
Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replac

Query: 'what airline does Sarah use?'
Found 3 matches:

  [airline] Prefers United Airlines for domestic flights (confidence: 0.90)
  [seating] Always requests aisle seats (confidence: 0.85)
  [hotel] Prefers Marriott hotels (confidence: 0.90)


## Vector Search: Synonyms Handled Automatically

The query "what airline does Sarah use?" never appears in any stored preference. But vector similarity finds the match because "I always fly United" and "what airline does Sarah use?" are semantically close. No keyword normalization, no synonym tables, no ontology maintenance.

In [ ]:
for query in ["what airline does Sarah use?", "hotel choice", "dietary needs", "where does she live?"]:
    results = await store.search(query, top_k=2)
    print(f"\nQuery: '{query}'")
    for r in results:
        print(f"  [{r['category']}] {r['preference']} (confidence: {r['confidence']:.2f}, score: {r['score']:.4f})")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nCALL db.index.vector.queryNodes('entity_embedding_idx', $limit, $embedding)\nYIELD node, score\nWHERE score >= $threshold\nRETURN node AS e, score\nORDER BY score DESC\n"
Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is 

Graph state: 3 entities, 3 preferences

Entities:
  [LOCATION] London
  [ORGANIZATION] Contoso
  [PERSON] Sarah Chen

Preferences:
  [hotel] Prefers Marriott hotels
  [airline] Prefers United Airlines for domestic flights
  [seating] Always requests aisle seats


## Supersession: Preferences Change Over Time

Sarah switches from Marriott to Hilton. Storing the new preference automatically retires the old one in the same category — `valid_to` is set on the Marriott document, `valid_from` on the Hilton document. Both remain for audit; only the current one appears in recall.

In [ ]:
# Sarah changes hotel preference
await store.add_preference("hotel_chain", "Prefers Hilton hotels", confidence=0.9)

# Current recall — only Hilton appears
results = await store.search("hotel preference", top_k=3)
print("Current recall:")
for r in results:
    print(f"  [{r['category']}] {r['preference']} | valid_to={r['valid_to']}")

# Full history — both Marriott (retired) and Hilton (current)
print("\nHotel chain history:")
for h in await store.history("hotel_chain"):
    status = "CURRENT" if h["valid_to"] is None else f"retired {h['valid_to'][:10]}"
    print(f"  {h['preference']} ({status})")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nCALL db.index.vector.queryNodes('preference_embedding_idx', $limit, $embedding)\nYIELD node, score\nWHERE score >= $threshold\nRETURN node AS p, score\nORDER BY score DESC\n"


Query: 'hotel choice'
  [hotel] Prefers Marriott hotels (confidence: 0.90)
  [airline] Prefers United Airlines for domestic flights (confidence: 0.90)
  [seating] Always requests aisle seats (confidence: 0.85)


## The Semantic-Memory Agent

The agent gets two memory tools:

| Tool | Purpose |
|------|---------|
| `learn_preference` | Store or update a preference from conversation |
| `recall_preferences` | Vector search over stored preferences |

The agent decides when to call these during conversation — it extracts the preference from natural language and stores it with a category and confidence.

In [ ]:
@tool
async def learn_preference(category: str, preference: str,
                           source_type: str = "user_assertion",
                           confidence: float = 0.8) -> str:
    """Store a personal preference. Categories: airline, hotel_chain, seating,
    dietary, home_city, loyalty, budget, etc. source_type: 'user_assertion'
    for explicit user statements, 'llm_inference' for derived conclusions."""
    doc = await store.add_preference(category, preference, confidence, source_type)
    sup = f" (superseded previous value)" if doc.get("superseded_by") is None and await store.count() > 6 else ""
    return f"Learned [{category}]: {preference} (confidence={confidence}){sup}"

@tool
async def recall_preferences(query: str) -> str:
    """Search stored preferences by meaning. Returns matching preferences
    with category, confidence, and state."""
    results = await store.search(query, top_k=5)
    if not results:
        return "No preferences found."
    lines = []
    for r in results:
        lines.append(f"[{r['category']}] {r['preference']} (confidence: {r['confidence']:.2f})")
    return "\n".join(lines)

print("Tools ready: learn_preference, recall_preferences")

In [ ]:
SEMANTIC_PROMPT = SYSTEM_PROMPT + """

You have access to Sarah Chen's (employee E001) long-term semantic memory.
- When the user reveals a preference, opinion, or personal fact, call learn_preference
  with an appropriate category and confidence (0.0-1.0).
- Before personalizing a recommendation, call recall_preferences to check what you know.
- Use source_type='user_assertion' for explicit statements, 'llm_inference' for derived conclusions.
- Always use user_id 'E001' for Sarah Chen.
"""

semantic_agent = Agent(
    client=client,
    name="SemanticMemoryAgent",
    instructions=SEMANTIC_PROMPT,
    tools=[search_flights, search_hotels, get_travel_policy,
           learn_preference, recall_preferences],
)
print(f"Agent ready: {semantic_agent.name}")

In [ ]:
session = AgentSession()

r1 = await semantic_agent.run(
    "I always stay at Marriott when I travel for work, and I need aisle seats on long flights.",
    session=session,
)
print("Turn 1:", r1.text)

r2 = await semantic_agent.run(
    "Can you find me flights to London and a good hotel there?",
    session=session,
)
print("\nTurn 2:", r2.text)

r3 = await semantic_agent.run(
    "Actually, I've switched to Hilton — their loyalty program is better for my travel pattern.",
    session=session,
)
print("\nTurn 3:", r3.text)

r4 = await semantic_agent.run(
    "So what do you know about my preferences now?",
    session=session,
)
print("\nTurn 4:", r4.text)

Semantic agent ready (5 tools)


## What's Actually Stored

Let's inspect the preference documents — note the supersession on hotel_chain (Marriott retired, Hilton current).

In [ ]:
for doc in await store.snapshot(include_deprecated=False):
    status = "CURRENT" if doc["valid_to"] is None else f"retired"
    print(f"  [{doc['category']:12s}] {doc['preference'][:45]:45s} | {doc['state']:11s} | {status}")

Sarah: Hi, I'm Sarah Chen (E001). I'm a Senior Engineer at Contoso.


AzureCliCredential.get_token_info failed: Failed to invoke the Azure CLI
AzureCliCredential.get_token_info failed: Failed to invoke the Azure CLI
Structured extraction failed (CredentialUnavailableError); falling back to plain LLM call
AzureCliCredential.get_token_info failed: Failed to invoke the Azure CLI
AzureCliCredential.get_token_info failed: Failed to invoke the Azure CLI
Function failed. Error: Failed to extract entities: Failed to invoke the Azure CLI
Function 'learn_from_conversation' raised an exception; returning an error result to the model. Set include_detailed_errors=True for the full detail. Exception: ExtractionError('Failed to extract entities: Failed to invoke the Azure CLI')


ChatClientException: ("<class 'agent_framework_foundry._chat_client.FoundryChatClient'> service failed to complete the prompt: Connection error.", APIConnectionError('Connection error.'))

## Comparison: Episodic vs Semantic Memory

| Dimension | Episodic Memory | Semantic Memory |
|---|---|---|
| **Stores** | Events, experiences | Facts, preferences, beliefs |
| **Schema** | `{event_type, description, details}` | `{category, preference, confidence, state, valid_from}` |
| **Query** | Filter by user + type | Vector similarity + property filters |
| **Contradictions** | Both versions stored, no priority | Supersession: old version retired, new version current |
| **Backend** | Cosmos DB (events container) | Cosmos DB (semantic-memory container) |
| **Reasoning** | Retrieve events, hope LLM connects | Direct lookup with embedding similarity |
| **Best for** | "What happened?" | "What do we know?" |

## Key Takeaways

1. **Semantic memory stores conclusions, not events.** "Prefers Marriott" is a structured fact with a category, confidence, and timestamp — not a raw event that needs inference.
2. **Vector embeddings solve synonym resolution.** "hotel choice", "where does she stay", and "accommodation preference" all match the same document. No ontology or relationship normalization needed.
3. **Supersession handles change.** When a preference changes, the old document is retired (`valid_to` set) and the new one becomes current. Both remain for audit. No ambiguity about which version is active.
4. **Cosmos DB is a natural fit.** Preference documents with vector search, partial updates for lifecycle properties, and native TTL — all in the same service as episodic memory.
5. **The agent decides what to store.** The model extracts preferences from conversation and calls `learn_preference` with a category and confidence. The storage layer handles versioning and deduplication.

## What Semantic Memory Still Cannot Do

Semantic memory stores facts and preferences. But it cannot:

- **Gate trust**: any write is treated as fact — there's no "candidate → provisional → trusted" lifecycle
- **Handle conflicting confidence**: two preferences at different confidence levels have no priority
- **Explain itself**: the agent knows *what* Sarah prefers but not *why* or *how confidently*
- **Bound growth**: over time, noise accumulates — there's no retention scoring or eviction

These require the **memory lifecycle** in Module 03.

## Next: Procedural Memory

The agent knows *what happened* (episodic) and *what Sarah prefers* (semantic). But it cannot execute a multi-step workflow like "book an international trip: check visa → insurance → flights → hotels → policy approval." That requires **procedural memory** — learned skills the agent can select and execute.

In [ ]:
await cosmos.close()
print("Cosmos client closed")